In [0]:
import os
import sys
import shutil
from pathlib import Path
import json

# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from DimMember.EDIProcessing.mapper import Mapper

In [0]:
def move_file(src: Path, dest_dir: Path) -> Path:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / src.name
    shutil.move(str(src), str(dest_file))
    return dest_file

In [0]:
def process_single_file(pending_file: Path, base_source_dir: Path) -> tuple:
    active_file = pending_file
    try:
        if not active_file.exists():
            raise FileNotFoundError(f"Input file missing: {active_file}")
        
        active_file = move_file(active_file, base_source_dir / "inprogress")

        # Parsing and mapping logic
        structured_json = EDIProcessor().parse(str(active_file))
        
        # Safe extraction from segments
        interchange = structured_json.get('interchange', {})
        client_id = interchange.get('sender_id', '').strip()
        file_id = interchange.get('control_number', '').strip()
        
        st_segment = structured_json.get('heading', {}).get('transaction_set_header_loop', {}).get('transaction_set_header_ST', {})
        layout_id = st_segment.get('transaction_set_identifier_code', '834').strip()
        
        # Generate output delivery
        output_file = ROOT_DIR / "temp" / layout_id / f"{active_file.stem}.csv"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        CSVConverter().converter(Mapper().map_member(structured_json), str(output_file))
        
        move_file(active_file, base_source_dir / "processed")
        print(f"Successfully processed: {active_file.name}")
        
        return client_id, file_id, layout_id, output_file
        
    except Exception as e:
        print(f"Failed processing {active_file.name}: {e}")
        if active_file.exists():
            move_file(active_file, base_source_dir / "failed")
        raise

In [0]:
def build_payloads(processed_files: list) -> tuple:
    """Generates payloads for both downstream notebooks in a single sweep."""
    process_list = []
    consolidation_list = []
    
    for f in processed_files:
        # 1. Process Payload Item
        process_list.append({
            "ClientID": f['client_id'],
            "FileID": f['file_id'],
            "FileName": f['csv_filename'],
            "ClientContainer": f"{ROOT_DIR}/temp/{f['layout_id']}/",
            "CurrentFolderPath": "",
            "ProcessedFolderPath": "/Volumes/claimsprocessing/bronze/member",
            "ColumnDelimiter": ",",
            "HasHeader": "true",
            "IgnoreHeader": "False",
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "SchemaFileName": "MemberSchema.json",
            "SchemaFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema",
            "TextQualifier": "\""
        })
        
        # 2. Consolidation Payload Item
        consolidation_list.append({
            "DataGroupTrackingID": f"TRACK_MEMBER_{f['layout_id']}_{f['file_id']}",
            "DataGroupMappingId": f"{f['layout_id']}MEMBER",
            "FileId": f['file_id'],
            "FileLayoutID": f['layout_id'],
            "FileLayoutDescription": f"Standard{f['layout_id']}",
            "CurrentContainer": "/Volumes/claimsprocessing/bronze/member",
            "CurrentFolderPath": "",
            "ConsolidatedMappingFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema/Consolidation",
            "ConsolidatedMappingFileName": "ConsolidationMember.json",
            "ConsolidatedLayerDataModelFilePath": f"{ROOT_DIR}/DimMember/Bronze/Schema/Consolidation/DataModels",
            "ConsolidatedLayerDataModel": "MemberDataModel.json",
            "ConsolidatedFolderPath": "/Volumes/claimsprocessing/bronze/member_consolidated"
        })
        
    return json.dumps({"FileIds": process_list}), json.dumps({"FileIds": consolidation_list})

In [0]:
def main():
    base_source_dir = ROOT_DIR / "source/834"
    pending_dir = base_source_dir / "pending"
    
    if not pending_dir.exists():
        return
        
    pending_files = [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]
    if not pending_files:
        print("No files found to process.")
        return

    processed_files = []
    for file_path in pending_files:
        try:
            c_id, f_id, l_id, out_path = process_single_file(file_path, base_source_dir)
            processed_files.append({
                'client_id': c_id, 'file_id': f_id, 'layout_id': l_id,
                'csv_path': str(out_path), 'csv_filename': out_path.name
            })
        except Exception as e:
            print(f"Skipping {file_path.name}: {e}")

    if not processed_files:
        return

    # Generate payloads
    process_payload, consolidation_payload = build_payloads(processed_files)
    
    notebook_base = f"{ROOT_DIR}/Shared/Notebooks"
    
    # Run Orchestration Chain
    try:
        print("\n=== Triggering FilesToProcess ===")
        dbutils.notebook.run(f"{notebook_base}/FilesToProcess", 600, {"ProcessedJSON": process_payload})
        
        print("\n=== Triggering LoopConsolidationFiles ===")
        dbutils.notebook.run(f"{notebook_base}/LoopConsolidationFiles", 600, {"ConsolidationJSON": consolidation_payload})
        
        print("\nPipeline execution sequence completed successfully.")
    except Exception as e:
        print(f"Downstream orchestration failed: {e}")
        raise

if __name__ == "__main__":
    main()

In [0]:
spark.sql("SELECT COUNT(*) FROM claimsprocessing.bronze.member_consolidated").show()